In [1]:
import pandas as pd, requests

API_KEY = "Smzd9wWG8SjNAK9xpUX5tAVgCfSOpjGSiQUJwfvG"
states = ["AL","GA","SC","TX","LA","FL","TN","MS","MA","RI","CT","NY","NJ","NH","MI","OH",
          "PA","IN","KY","IL","MD","VA","NC","AR","AZ","CO","NM","NV","KS","IA","MN",
          "SD","CA","OR","WA","UT"]

records = []
for st in states:
    params = {
        "api_key": API_KEY,
        "frequency": "weekly",
        "data[0]": "value",
        "facets[duoarea][]": "S"+st,   # state code
        "facets[product][]": "EPMR",   # regular gasoline
        "facets[process][]": "PTE",    # retail
        "start": "2024-01-01",
        "end": "2025-09-14",
        "length": 5000,
    }
    r = requests.get("https://api.eia.gov/v2/petroleum/pri/gnd/data/", params=params)
    r.raise_for_status()
    data = r.json().get("response", {}).get("data", [])
    for d in data:
        records.append({
            "dt": d["period"],
            "state_name": st,
            "gas_price_usd_per_gallon": float(d["value"]),
        })

gas = pd.DataFrame(records)
gas["dt"] = pd.to_datetime(gas["dt"])
gas = gas.sort_values(["state_name","dt"])
# upsample to daily with forward-fill
start, end = pd.Timestamp("2024-03-14"), pd.Timestamp("2025-09-14")
frames = []
for st, grp in gas.groupby("state_name"):
    idx = pd.date_range(start, end)
    filled = grp.set_index("dt").reindex(idx, method="ffill").rename_axis("dt").reset_index()
    filled["state_name"] = st
    frames.append(filled)
gas_daily = pd.concat(frames, ignore_index=True)
gas_daily.to_csv("data/external/gas_price_state_daily.csv", index=False)
